<a href="https://colab.research.google.com/github/HariniRJ/bio-lab/blob/main/grey_wolf_opti_%2C1BM24CS364_LAB_PROGRAM_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np

# ==========================================
# 1. PROBLEM SETUP & PARAMETERS
# ==========================================
# Problem Parameters
NUM_PATIENTS = 5
NUM_BEDS = 2

# Define Patient Characteristics (Synthetic Data)
# Priorities: 2 = Emergency (High Priority), 1 = Regular (Normal Priority)
np.random.seed(42)
patient_priorities = np.random.choice([1, 2], size=NUM_PATIENTS, p=[0.6, 0.4])
patient_treatment_times = np.random.randint(1, 6, size=NUM_PATIENTS)  # Treatment duration between 1 to 5 hours

# GWO Parameters
NUM_WOLVES = 20
MAX_ITER = 100

print("--- Patient Admission Data ---")
for i in range(NUM_PATIENTS):
    status = "Emergency" if patient_priorities[i] == 2 else "Regular"
    print(f"Patient {i}: Status = {status}, Duration = {patient_treatment_times[i]} hours")
print("-" * 30)

# ==========================================
# 2. FITNESS FUNCTION
# ==========================================
def calculate_fitness(position):
    """
    Evaluates the allocation of beds. Decodes continuous wolf positions into
    discrete bed indexes, orders queues by emergency priority, and computes
    total weighted waiting times.
    """
    # Convert continuous position values into discrete bed indices [0, NUM_BEDS - 1]
    bed_assignments = np.clip(np.round(position), 0, NUM_BEDS - 1).astype(int)

    total_cost = 0

    # Calculate queue times for each individual bed
    for bed_idx in range(NUM_BEDS):
        # Identify patients assigned to this specific bed
        assigned_patients = np.where(bed_assignments == bed_idx)[0]
        if len(assigned_patients) == 0:
            continue

        # Priority Queue: Sort patients by emergency level (highest priority first)
        sorted_patients = sorted(assigned_patients, key=lambda x: (-patient_priorities[x], x))

        current_wait_clock = 0
        for p_id in sorted_patients:
            # Apply a high penalty factor (15x) for keeping emergency patients waiting
            weight = 15 if patient_priorities[p_id] == 2 else 1
            total_cost += current_wait_clock * weight

            # Accumulated time before the next patient can use the bed
            current_wait_clock += patient_treatment_times[p_id]

    return total_cost

# ==========================================
# 3. GREY WOLF OPTIMIZER (GWO) ENGINE
# ==========================================
# Initialize population positions randomly within the boundaries of available beds
positions = np.random.uniform(0, NUM_BEDS - 1, size=(NUM_WOLVES, NUM_PATIENTS))

# Initialize Leader Wolf Hierarchies
alpha_pos = np.zeros(NUM_PATIENTS)
beta_pos = np.zeros(NUM_PATIENTS)
delta_pos = np.zeros(NUM_PATIENTS)

alpha_score = float('inf')
beta_score = float('inf')
delta_score = float('inf')

# Main Optimization Loop
for iteration in range(MAX_ITER):
    for i in range(NUM_WOLVES):
        # Boundary handling: Keep wolves within valid bed boundaries
        positions[i] = np.clip(positions[i], 0, NUM_BEDS - 1)

        # Evaluate current fitness score
        fitness = calculate_fitness(positions[i])

        # Update Hierarchy rankings (Alpha, Beta, Delta)
        if fitness < alpha_score:
            delta_score = beta_score
            delta_pos = beta_pos.copy()
            beta_score = alpha_score
            beta_pos = alpha_pos.copy()
            alpha_score = fitness
            alpha_pos = positions[i].copy()

        elif fitness < beta_score:
            delta_score = beta_score
            delta_pos = beta_pos.copy()
            beta_score = fitness
            beta_pos = positions[i].copy()

        elif fitness < delta_score:
            delta_score = fitness
            delta_pos = positions[i].copy()

    # Linearly decrease adaptive parameter 'a' from 2 down to 0
    a = 2 - iteration * (2 / MAX_ITER)

    # Update positions of all search agents (wolves)
    for i in range(NUM_WOLVES):
        for j in range(NUM_PATIENTS):

            # Equations relating to Alpha exploration
            r1, r2 = np.random.rand(), np.random.rand()
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
            X1 = alpha_pos[j] - A1 * D_alpha

            # Equations relating to Beta exploration
            r1, r2 = np.random.rand(), np.random.rand()
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = abs(C2 * beta_pos[j] - positions[i, j])
            X2 = beta_pos[j] - A2 * D_beta

            # Equations relating to Delta exploration
            r1, r2 = np.random.rand(), np.random.rand()
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = abs(C3 * delta_pos[j] - positions[i, j])
            X3 = delta_pos[j] - A3 * D_delta

            # The new position is the average position center of the 3 leading wolves
            positions[i, j] = (X1 + X2 + X3) / 3

# ==========================================
# 4. FINAL DECODING & DISPLAY RESULTS
# ==========================================
best_allocation = np.clip(np.round(alpha_pos), 0, NUM_BEDS - 1).astype(int)

print("\n=== OPTIMAL PATIENT ALLOCATION FOUND ===")
final_bed_queues = {b: [] for b in range(NUM_BEDS)}

for p_id, b_id in enumerate(best_allocation):
    final_bed_queues[b_id].append(p_id)

for bed_idx in range(NUM_BEDS):
    print(f"\n[Bed Class {bed_idx + 1}] Queue Schedule:")
    queue = final_bed_queues[bed_idx]
    if not queue:
        print("  -> Empty (No patients allocated)")
        continue

    # Re-apply sorting to view correct timelines
    sorted_queue = sorted(queue, key=lambda x: (-patient_priorities[x], x))

    cumulative_wait = 0
    for p_id in sorted_queue:
        status = "EMERGENCY" if patient_priorities[p_id] == 2 else "Regular"
        print(f"  - Patient {p_id:2d} ({status:9s}) -> Waits: {cumulative_wait:2d} hrs | Treatment: {patient_treatment_times[p_id]} hrs")
        cumulative_wait += patient_treatment_times[p_id]

print(f"\nFinal Optimized Fitness Objective Score: {alpha_score}")


--- Patient Admission Data ---
Patient 0: Status = Regular, Duration = 3 hours
Patient 1: Status = Emergency, Duration = 3 hours
Patient 2: Status = Emergency, Duration = 3 hours
Patient 3: Status = Regular, Duration = 5 hours
Patient 4: Status = Regular, Duration = 4 hours
------------------------------

=== OPTIMAL PATIENT ALLOCATION FOUND ===

[Bed Class 1] Queue Schedule:
  - Patient  2 (EMERGENCY) -> Waits:  0 hrs | Treatment: 3 hrs
  - Patient  0 (Regular  ) -> Waits:  3 hrs | Treatment: 3 hrs
  - Patient  3 (Regular  ) -> Waits:  6 hrs | Treatment: 5 hrs

[Bed Class 2] Queue Schedule:
  - Patient  1 (EMERGENCY) -> Waits:  0 hrs | Treatment: 3 hrs
  - Patient  4 (Regular  ) -> Waits:  3 hrs | Treatment: 4 hrs

Final Optimized Fitness Objective Score: 12
